# Notebook Overview

Purpose:

*   Transform Bronze Transformation data into a standardized Silver dataset.
*   Validate transaction records against documented data-quality rules.
*   Separate accepted and quarantined records.
*   Persist outputs and reconcile quality metrics.

### Configuration / Paths

In [88]:
from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"
BRONZE_TRANSACTION_PATH = f"{FILE_PATH}/bronze/transactions"
QUARANTINE_TRANSACTION_PATH = f"{FILE_PATH}/quarantine/transactions"
SILVER_TRANSACTION_PATH = f"{FILE_PATH}/silver/transactions"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Spark Session

In [89]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName('transaction-silver').getOrCreate()
spark.version

'4.0.4'

### Load Data from Bronze

In [90]:
def load_data(FILE_PATH):
  try:
    df_product = spark.read.parquet(FILE_PATH).withColumn(
        '_source_file', F.lit(F.input_file_name())
    )
    print('Data Load Successful')
    return df_product

  except Exception as e:
    raise RuntimeError(f"Data Load Failed: {str(e)}")

df_transaction = load_data(BRONZE_TRANSACTION_PATH)

Data Load Successful


### Row Count and Bronze Transaction Schema

In [91]:
def schema_and_count(df):
  print('\nRow Count:', df.count())
  print('Schema')
  print('======')
  df.printSchema()

schema_and_count(df_transaction)


Row Count: 50000
Schema
root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = false)



### Validate the Bronze Schema

In [92]:
def validate_transaction_schema(df):
  try:
    expected_columns = ['transaction_id',
                        'order_id',
                        'product_id',
                        'transaction_date',
                        'quantity',
                        'unit_price',
                        'discount_amount',
                        'tax_amount',
                        'transaction_amount',
                        'payment_status',
                        '_ingestion_timestamp',
                        '_source_file'
                        ]

    df_column = df.columns

    missing_cols = list(set(expected_columns) - set(df_column))

    expected_schema = {
          'transaction_id': 'string',
          'order_id': 'string',
          'product_id': 'string',
          'transaction_date': 'string',
          'quantity': 'string',
          'unit_price': 'string',
          'discount_amount': 'double',
          'tax_amount': 'double',
          'transaction_amount' : 'double',
          'payment_status': 'string',
          '_ingestion_timestamp': 'timestamp',
          '_source_file': 'string'
      }

    actual_types = dict(df.dtypes)

    type_missmatch = []
    for col, expected_type in expected_schema.items():
      if col in actual_types and actual_types[col] != expected_type:
        type_missmatch.append(f"Column '{col}' expected '{expected_type}', but got '{actual_types[col]}'")

    error_messages = []
    if missing_cols:
      error_messages.append(f'Missing_cols: {missing_cols}')
    if type_missmatch:
      error_messages.append(f'Type Missmatch: {type_missmatch}')

    if error_messages:
      raise ValueError("Transaction Schema Validation Failed:\n" + "\n".join(error_messages))
    print("Schema Validation passed successfully!")

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_transaction_schema': {str(e)}") from e

### Preserve Audit Information

In [93]:
def preserve_audit(df):
  df_preserve = df
  df_preserve = df.withColumn(
      '_original_order_id',
      F.col('order_id')
      ).withColumn(
          '_original_product_id',
          F.col('product_id')
      ).withColumn(
          '_original_transaction_date',
          F.col('transaction_date')
      ).withColumn(
          '_original_quantity',
          F.col('quantity')
      ).withColumn(
          '_original_unit_price',
          F.col('unit_price')
      ).withColumn(
          '_original_discount_amount',
          F.col('discount_amount')
      ).withColumn(
          '_original_tax_amount',
          F.col('tax_amount')
      ).withColumn(
          '_original_transaction_amount',
          F.col('transaction_amount')
      )

  return df_preserve

### Standardize String Columns

In [94]:
def standardize_transaction_strings(df):
  try:
    string_cols = [
        'transaction_id',
        'order_id',
        'product_id',
        'transaction_date',
        'quantity',
        'unit_price',
        'payment_status',
        '_source_file'
      ]

    df_clean = df

    for c in string_cols:
      df_clean = df_clean.withColumn(
          c,
          F.when(F.trim(F.lower(F.col(c))).isin('n/a', 'na', 'unknown', 'null', ''), None).otherwise(F.trim(F.col(c)))
      )

    return df_clean
  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'standardize_transaction_strings': {str(e)}") from e

### Cast Types

In [95]:
from pyspark.sql.types import DecimalType, DateType, IntegerType

def cast_transaction_types(df):
  try:
    if 'quantity' in df.columns:
      df_typed = df.withColumn('quantity', F.col('quantity').try_cast(IntegerType()))

    if 'transaction_date' in df.columns:
      df_typed = df_typed.withColumn('transaction_date', F.col('transaction_date').try_cast(DateType()))

    if 'unit_price' in df.columns:
      df_typed = df_typed.withColumn('unit_price', F.col('unit_price').try_cast(DecimalType(10,2)))

    if 'discount_amount' in df.columns:
      df_typed = df_typed.withColumn('discount_amount', F.col('discount_amount').try_cast(DecimalType(10,2)))

    if 'tax_amount' in df.columns:
      df_typed = df_typed.withColumn('tax_amount', F.col('tax_amount').try_cast(DecimalType(10,2)))

    if 'transaction_amount' in df.columns:
      df_typed = df_typed.withColumn('transaction_amount', F.col('transaction_amount').try_cast(DecimalType(10,2)))


    return df_typed

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'cast_transaction_types': {str(e)}") from e

In [96]:
from pyspark.sql.types import DecimalType, DateType, IntegerType
df_transaction.withColumn(
    'unit_price',
    F.col('unit_price').try_cast(DecimalType(10,2))
).withColumn(
    'quantity',
    F.col('quantity').try_cast(IntegerType())
).withColumn(
    'tax_amount',
    F.col('tax_amount').try_cast(DecimalType(10,2))
).withColumn(
    'discount_amount',
    F.col('discount_amount').try_cast(DecimalType(10,2))
).withColumn(
    'transaction_amount',
    F.col('transaction_amount').try_cast(DecimalType(10,2))
).select(
    'transaction_amount',
    F.expr("(quantity * unit_price) + tax_amount - discount_amount ").alias('calculated'),
    (F.col('transaction_amount') - F.col('calculated')).alias('diff')
).filter("diff >= 0.5").show()

+------------------+----------+---------+
|transaction_amount|calculated|     diff|
+------------------+----------+---------+
|            724.25|     53.65|   670.60|
|            251.88|   -239.12|   491.00|
|           1464.87|   1355.36|   109.51|
|         999999.99|    254.36|999745.63|
+------------------+----------+---------+



### Validate Transaction Records

In [97]:
def validate_transaction_records(df):
  status = ['settled', 'refunded', 'failed', 'captured', 'pending']
  try:
    df_validated = df.withColumn(
        'is_valid',
        F.when(F.col('transaction_id').isNull(), 'NULL or Missing Transaction ID')
        .when(F.col('order_id').isNull(), 'NULL or Missing Order ID')

        .when(F.col('product_id').isNull(), 'NULL or Missing Product ID')

        .when(F.col('transaction_date').isNull(), 'Invalid or NULL Date')

        .when((F.col('quantity').isNull()) | (F.col('quantity') < 0) | (F.col('quantity') == 0), 'Invalid or Missing Quantity')

        .when((F.col('unit_price').isNull()) | (F.col('unit_price') < 0) | (F.col('unit_price') == 0), 'Invalid or Missing Price')

        .when((F.col('discount_amount').isNull()) | (F.col('discount_amount') < 0), 'Negative or NULL amount')

        .when((F.col('tax_amount').isNull()) | (F.col('tax_amount') < 0), 'Negative or NULL tax amount')

        .when(~F.lower(F.col('payment_status')).isin(status), 'Unknownstatus')

        .when(F.col('transaction_amount').isNull(), 'Missing or NULL amount')

        .when(
        F.col("transaction_amount") - ((F.col("quantity") * F.col("unit_price"))
            + F.col("tax_amount") - F.col("discount_amount")) >= 0.5, "Transaction Amount Mismatch")
    )

    return df_validated
  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_transaction_records': {str(e)}") from e

### Handle Duplicate Transaction IDs

In [98]:
from pyspark.sql import Window as W

def handle_duplicate_ids(df):
  try:
    window_spec = W.partitionBy('transaction_id').orderBy(F.col('_ingestion_timestamp').desc())

    df_ranked = df.withColumn(
        'rank',
        F.row_number().over(window_spec)
    )

    df_mark = df_ranked.withColumn(
        'is_valid',
        F.coalesce(
            F.col('is_valid'),
            F.when(F.col('rank') > 1, 'Duplicate Transaction ID')
        )
    ).drop('rank')

    return df_mark

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'handle_duplicate_ids': {str(e)}") from e

### Split Valid and Quarantined Records

In [99]:
def split_transaction_by_quality(df):
  try:
    df_valid = df.filter(F.col('is_valid').isNull()).drop('is_valid', '_original_order_id', '_original_product_id', '_original_transaction_date', '_original_quantity', '_original_tax_amount', '_original_discount_amount', 'original_transaction_amount', '_original_unit_price')

    df_quarantine = df.filter(F.col('is_valid').isNotNull())

    return df_valid, df_quarantine

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'split_transaction_by_quality': {str(e)}") from e

### Orchestrate the Transaction Transformation

In [100]:
def transform_transaction(df):
  try:
    print('===============================')
    print('TRANSFORMING DATA')
    print('===============================')
    print('Start: Validating Bronze Schema')
    validate_transaction_schema(df)
    print('Finished: Validaing Bronze Schema')
    print('Start: Preserving Data for Audit')
    preserved_df = preserve_audit(df)
    print('Finished: Preserving Data for Audit')
    print('Start: Standardizing String Columns')
    standardize_df = standardize_transaction_strings(preserved_df)
    print('Finished: Standardizing String Columns')
    print('Start: Cast type Column')
    casted_df = cast_transaction_types(standardize_df)
    print('Finished: Cast type Column')
    print('Start: Validating Transactions Record')
    validated_df = validate_transaction_records(casted_df)
    print('Finished: Validating Transactions Record')
    print('Start: Handling and Marking duplicate id')
    df_mark = handle_duplicate_ids(validated_df)
    print('Finished: Handling and Marking duplicate id')
    print('Spiliting and Returning Validated and Quanrantine data')
    print('===============================')
    print('TRANSFORMING COMPLETE')
    print('===============================')
    return split_transaction_by_quality(df_mark)

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Failed inside 'transform_transaction': {str(e)}") from e

### **Transforming Transaction Data**

In [101]:
valid_df, quarantine_df = transform_transaction(df_transaction)

TRANSFORMING DATA
Start: Validating Bronze Schema
Schema Validation passed successfully!
Finished: Validaing Bronze Schema
Start: Preserving Data for Audit
Finished: Preserving Data for Audit
Start: Standardizing String Columns
Finished: Standardizing String Columns
Start: Cast type Column
Finished: Cast type Column
Start: Validating Transactions Record
Finished: Validating Transactions Record
Start: Handling and Marking duplicate id
Finished: Handling and Marking duplicate id
Spiliting and Returning Validated and Quanrantine data
TRANSFORMING COMPLETE


### Calculate quality metrics and reconcile

In [102]:
def generate_pipeline_metrics(df_bronze, df_valid, df_quarantine):
    print("\n" + "=" * 65)
    print(" PIPELINE RECONCILIATION & QUALITY METRICS REPORT")
    print("=" * 65)

    input_count = df_bronze.count()
    silver_count = df_valid.count()
    quarantine_count = df_quarantine.count()

    print("\n[1] RECONCILIATION SUMMARY")
    print(f"  • Total Bronze Input Count      : {input_count:,}")
    print(f"  • Accepted Silver Count         : {silver_count:,}")
    print(f"  • Quarantined Count             : {quarantine_count:,}")

    sum_outputs = silver_count + quarantine_count
    is_balanced = (input_count == sum_outputs)
    print(f"  • Reconciliation Balance Check  : {'PASSED (Match)' if is_balanced else 'FAILED (Mismatch!)'}")
    if not is_balanced:
        print(f"    WARNING: Input ({input_count:,}) != Silver ({silver_count:,}) + Quarantine ({quarantine_count:,})")

    print("\n[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)")
    if quarantine_count > 0:
        print("  Breakdown of reasons records were quarantined:")
        df_quarantine.groupBy("is_valid").count().orderBy(F.col("count").desc()).show(truncate=False)
    else:
        print("  • No quarantined records found. 100% clean dataset!")

    print("\n[3] NULL COUNTS IN SILVER LAYER")
    print("  (Tracking optional profile fields alongside required ones)")
    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_valid.columns]
    df_valid.select(null_exprs).show(truncate=False)

    print("\n[4] UNIQUENESS CHECK (transaction_id in Silver)")
    distinct_orders = df_valid.select("transaction_id").distinct().count()
    is_unique = (silver_count == distinct_orders)
    print(f"  • Total Silver Rows             : {silver_count:,}")
    print(f"  • Unique Transaction IDs           : {distinct_orders:,}")
    print(f"  • Uniqueness Constraint         : {'PASSED (No duplicate IDs in Silver)' if is_unique else 'FAILED (Duplicates present!)'}")

    print("=" * 65)

generate_pipeline_metrics(df_transaction, valid_df, quarantine_df)


 PIPELINE RECONCILIATION & QUALITY METRICS REPORT

[1] RECONCILIATION SUMMARY
  • Total Bronze Input Count      : 50,000
  • Accepted Silver Count         : 49,990
  • Quarantined Count             : 10
  • Reconciliation Balance Check  : PASSED (Match)

[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)
  Breakdown of reasons records were quarantined:
+---------------------------+-----+
|is_valid                   |count|
+---------------------------+-----+
|Invalid or Missing Quantity|3    |
|Invalid or NULL Date       |2    |
|Negative or NULL tax amount|1    |
|Transaction Amount Mismatch|1    |
|Negative or NULL amount    |1    |
|Invalid or Missing Price   |1    |
|Duplicate Transaction ID   |1    |
+---------------------------+-----+


[3] NULL COUNTS IN SILVER LAYER
  (Tracking optional profile fields alongside required ones)
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+-----------

### Adding Metadata

In [104]:
from pyspark.sql.types import TimestampType


# Adding metadata to Quarantine Dataset
quarantine_df = quarantine_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

# Adding metadata to Valid Dataset
valid_df = valid_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

silver_transaction_df = valid_df.select(
    'transaction_id',
    'order_id',
    'product_id',
    'transaction_date',
    'quantity',
    'unit_price',
    'discount_amount',
    'tax_amount',
    'transaction_amount',
    'payment_status',
    '_silver_processed_timestamp',
    '_source_file'
)

silver_transaction_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)
 |-- transaction_amount: decimal(10,2) (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = true)



### ***Write into Silver***

In [106]:
try:
  print('INFO: Writing records to Quarantine layer...')
  # Write Quarantine Data
  quarantine_df.write.mode('overwrite').parquet(QUARANTINE_TRANSACTION_PATH)
  print('SUCCESS: Quarantine dataset written successfully.')

  print('INFO: Writing clean records to Silver layer...')
  # Write Siver Data into Layer
  silver_transaction_df.write.mode('overwrite').parquet(SILVER_TRANSACTION_PATH)
  print('SUCCESS: Silver dataset written successfully.')

except Exception as e:
    raise RuntimeError(f"[Pipeline Error] Failed to write Silver dataset: {str(e)}") from e


INFO: Writing records to Quarantine layer...
SUCCESS: Quarantine dataset written successfully.
INFO: Writing clean records to Silver layer...
SUCCESS: Silver dataset written successfully.


### Read Back and Verify

In [107]:
df = spark.read.parquet(SILVER_TRANSACTION_PATH)

quarantine_df = spark.read.parquet(QUARANTINE_TRANSACTION_PATH)

schema_and_count(df)
schema_and_count(quarantine_df)


Row Count: 49990
Schema
root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)
 |-- transaction_amount: decimal(10,2) (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


Row Count: 10
Schema
root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)